In [4]:
from tqdm import tqdm
from omm.omm import ProteinImplicit
import mdtraj as md 
import numpy as np
import torch
import subprocess
from einops import rearrange, reduce, repeat
import os


In [5]:
parent_dir = "/global/cfs/cdirs/m4235/boltz_files/bba/boltz_inference/boltz_results_bba_inference_step_1.6/predictions/bba/"
protein_name = "bba"


In [6]:
# load the diffusion structures as a concatenated pdb

diffusion_structures = []
file_list = os.listdir(parent_dir)
file_list = sorted(file_list, key=lambda x: int(x.split("_")[-1].split(".")[0]))
print(file_list)


['bba_model_0.pdb', 'bba_model_1.pdb', 'bba_model_2.pdb', 'bba_model_3.pdb', 'bba_model_4.pdb', 'bba_model_5.pdb', 'bba_model_6.pdb', 'bba_model_7.pdb', 'bba_model_8.pdb', 'bba_model_9.pdb', 'bba_model_10.pdb', 'bba_model_11.pdb', 'bba_model_12.pdb', 'bba_model_13.pdb', 'bba_model_14.pdb', 'bba_model_15.pdb', 'bba_model_16.pdb', 'bba_model_17.pdb', 'bba_model_18.pdb', 'bba_model_19.pdb', 'bba_model_20.pdb', 'bba_model_21.pdb', 'bba_model_22.pdb', 'bba_model_23.pdb', 'bba_model_24.pdb', 'bba_model_25.pdb', 'bba_model_26.pdb', 'bba_model_27.pdb', 'bba_model_28.pdb', 'bba_model_29.pdb', 'bba_model_30.pdb', 'bba_model_31.pdb', 'bba_model_32.pdb', 'bba_model_33.pdb', 'bba_model_34.pdb', 'bba_model_35.pdb', 'bba_model_36.pdb', 'bba_model_37.pdb', 'bba_model_38.pdb', 'bba_model_39.pdb', 'bba_model_40.pdb', 'bba_model_41.pdb', 'bba_model_42.pdb', 'bba_model_43.pdb', 'bba_model_44.pdb', 'bba_model_45.pdb', 'bba_model_46.pdb', 'bba_model_47.pdb', 'bba_model_48.pdb', 'bba_model_49.pdb', 'bba_mode

In [7]:
# only run this once!

for filename in tqdm(file_list):
    if filename.endswith(".pdb"):
        diffusion_structures.append(md.load(f"{parent_dir}/{filename}"))
diffusion_structures = md.join(diffusion_structures)
diffusion_structures.save(f"{parent_dir}/all_diffusion_structures.pdb")

100%|██████████| 2000/2000 [00:10<00:00, 182.14it/s]


In [ ]:
# otherwise just load the concatenated pdb

diffusion_structures = md.load(f"{parent_dir}/all_diffusion_structures.pdb")

In [8]:
def compute_force_from_traj(
    traj: md.Trajectory, 
    amber_filename:str, 
    num_relax_steps:int=0, 
    temperature:float=300
):
    """Compute the heavy atom forces at every trajectory frame.

    Parameters
    ----------
    traj : md.Trajectory
        Trajectory to compute forces for.
    amber_filename : str
        Name of the .prmtop and .(inp)crd files of the system.
    
    Notes
    -----
    Assumes kT units with T = 300 Kelvin unless otherwise specified.
    """    
    simulation_args = {
        "temperature": temperature, 
        "temperature_units": "kelvin", 
        "friction": 100.0, "dt": 0.00002, 
        "time_units": "picoseconds", 
        "prior_weight": None, 
        "integrator_to_use": "overdamped", 
        "do_energy_minimization": False, 
        "chk_freq": 10000000, 
        "device": "CPU", 
        "fix":"backbone"
    }
    solvent_args = {
        "implicit_solvent": "OBC2", 
        "implicit_solvent_kappa": 0.1, 
        "implicit_solvent_kappa_length_units": "nanometer"
    }

    p = ProteinImplicit(
        filename = amber_filename, chk=0,
        simulation_args=simulation_args,
        solvent_args=solvent_args,
        save_filename = f"./"
        )

    gen_positions = traj.xyz # mdtraj saves in nanometers by default
    final_forces = []

    for i, position in tqdm(enumerate(gen_positions), mininterval=10):
        # Need to somehow add hydrogens to each position tensor. 
        pos, pe, ke, forces = p.relax_energies(
            10 * position, # convert to angstroms
            velocities=True,
            num_relax_steps=num_relax_steps, # 0 relax steps means no relaxation, just get energies
            length_units="angstroms",
            time_units="picoseconds",
            energy_units="kilocalories_per_mole",
            # energy_units="kT"
        )
        final_forces.append(forces)

    return np.stack(final_forces, axis=0)

In [9]:
# run the amber files to add missing hydrogens and OXT and save prmtop and crd files

tleap_input = f"""
    source leaprc.ff99SBxildn
    protein = loadPDB {parent_dir}/{file_list[0]}
    check protein
    savePdb protein amber_{protein_name}_single.pdb
    saveAmberParm protein amber_{protein_name}_single.prmtop amber_{protein_name}_single.crd
    quit
    """

result = subprocess.run(
    ["tleap", "-f", "-"],  # "-" means read from stdin
    input=tleap_input.encode(),
    stdout=subprocess.DEVNULL,  # suppress normal output
    check=True
)

# diffusion iid inference traj
traj_for_forces_path = f"{parent_dir}/all_diffusion_structures.pdb"

tleap_input = f"""
    source leaprc.ff99SBxildn
    protein = loadPDB {traj_for_forces_path}
    saveAmberParm protein amber_{protein_name}_traj.prmtop amber_{protein_name}_traj.crd
    quit
    """

result = subprocess.run(
    ["tleap", "-f", "-"],  # "-" means read from stdin
    input=tleap_input.encode(),
    stdout=subprocess.DEVNULL,  # suppress normal output
    check=True
)

# turn the prmtop and crd files into a pdb file with hydrogens
subprocess.run(f"ambpdb -p amber_{protein_name}_traj.prmtop -c amber_{protein_name}_traj.crd > amber_{protein_name}_traj.pdb", shell=True, check=True)

CompletedProcess(args='ambpdb -p amber_bba_traj.prmtop -c amber_bba_traj.crd > amber_bba_traj.pdb', returncode=0)

In [10]:
# the fixed trajectory in amber format needs to be reshaped

amber_traj = md.load(f"amber_{protein_name}_traj.pdb")
amber_top = md.load(f"amber_{protein_name}_single.pdb").topology
print(f"{amber_traj.xyz.shape = }")

amber_pos = rearrange(amber_traj.xyz, "1 (frame atom) dim -> frame atom dim", 
                      frame=diffusion_structures.xyz.shape[0],)
fixed_amber_traj = md.Trajectory(amber_pos, amber_top).center_coordinates()
fixed_amber_traj.save(f"fixed_amber_{protein_name}_traj.pdb")
print(f"{fixed_amber_traj.xyz.shape = }")

amber_traj.xyz.shape = (1, 1008000, 3)
fixed_amber_traj.xyz.shape = (2000, 504, 3)


In [ ]:
# finally, compute the forces

gt_forces = compute_force_from_traj(traj=fixed_amber_traj, amber_filename="/global/homes/d/dunne/boltz-likelihoods/notebooks/amber_bba_single", num_relax_steps=0, temperature=300)
heavy_atom_indices = amber_top.select("not element H and not name OXT")
gt_forces = torch.tensor(gt_forces[:, heavy_atom_indices, :])
print(torch.mean(torch.norm(gt_forces, dim=-1)))


/global/u2/d/dunne/boltz-likelihoods/src/omm/omm.py:271: UserWarning: Check all Implicit solvent parameters (e.g. solvent)
  warnings.warn("Check all Implicit solvent parameters (e.g. solvent)")
/global/homes/d/dunne/boltz-likelihoods/.venv/lib64/python3.11/site-packages/openmm/app/internal/amber_file_parser.py:1168: UserWarning: Non-optimal GB parameters detected for GB model OBC2
  warnings.warn(
822it [00:30, 27.16it/s]